# Artificial Neural Network 

An Artificial Neural Network (ANN) is a computer system inspired by the human brain that learns to find patterns in data.

## What is an ANN?

It uses connected pieces called "artificial neurons" arranged in layers.

Data enters the first layer, passes through hidden middle layers, and produces an answer in the final layer.

The system compares its answers to correct examples and adjusts its internal settings to make fewer mistakes next time.

## What is it used for the most?

**Image recognition:** Spotting faces or objects in photos.

**Speech-to-text:** Converting spoken words into typed text for voice assistants.

**Language translation:** Changing text from one language to another.

**Forecasting:** Predicting future numbers like stock prices or weather.

## Applications in Retail

**Product Recommendations:** Suggesting items you might like based on what you and other shoppers bought before.

**Inventory Forecasting:** Predicting how much stock a store needs to buy for upcoming holidays or seasons.

**Dynamic Pricing:** Adjusting product prices automatically based on demand, competitor prices, and shopping trends.

**Customer Service Chatbots:** Answering customer questions quickly on shopping websites.

**Fraud Detection:** Spotting unusual or fake credit card purchases instantly.

# Import main Libraries

In [25]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import tensorflow as tf
from tensorflow import keras

# Load dataset



In [26]:
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


# Dtypes

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


# Feature' Selection

X will identify some passegner specs

Y is the binary outcome that identifies the survival of the passenger

In [28]:
features = [
    "age",
    "fare",
    "sibsp",
    "parch",
    "pclass",
    "sex",
    "embarked"
]

X = df[features]
y = df["survived"]

# Identification of Numerical and Categorical

In [29]:
numeric_features = [
    "age",
    "fare",
    "sibsp",
    "parch",
    "pclass"
]

categorical_features = [
    "sex",
    "embarked"
]

# Train Test Split

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
# Note we could use also a different formula for the splitting

# train_df = df.sample(frac=0.8, random_state=42)

# test_df = df.drop(train_df.index)

# Missing Values

In [32]:
X_train[numeric_features].isna().sum()

age       137
fare        0
sibsp       0
parch       0
pclass      0
dtype: int64

# We use the simple imputer as fill nan strategy

In [33]:
numeric_imputer = SimpleImputer(
    strategy="median"
)

In [34]:
X_train_num = pd.DataFrame(
    numeric_imputer.fit_transform(X_train[numeric_features]),
    columns=numeric_features,
    index=X_train.index
)
X_train_num[numeric_features].isna().sum()

age       0
fare      0
sibsp     0
parch     0
pclass    0
dtype: int64

# We do it again on the test

In [35]:
X_test[numeric_features].isna().sum() # no need of imputer but in case

age       40
fare       0
sibsp      0
parch      0
pclass     0
dtype: int64

In [36]:
X_test_num = pd.DataFrame(
    numeric_imputer.transform(X_test[numeric_features]),
    columns=numeric_features,
    index=X_test.index
)
X_test_num[numeric_features].isna().sum()

age       0
fare      0
sibsp     0
parch     0
pclass    0
dtype: int64

# Missing Values on Categories

In [37]:
categorical_imputer = SimpleImputer(
    strategy="most_frequent"
)

In [38]:
X_train[categorical_features].isna().sum()

sex         0
embarked    2
dtype: int64

In [39]:
X_train_cat = pd.DataFrame(categorical_imputer.fit_transform(
    X_train[categorical_features]
), columns = categorical_features,
index = X_train.index
)
X_train_cat[categorical_features].isna().sum()

sex         0
embarked    0
dtype: int64

In [40]:
X_test_cat = pd.DataFrame(categorical_imputer.transform(
    X_test[categorical_features]
), columns = categorical_features,
index = X_test.index
)
X_test_cat[categorical_features].isna().sum()

sex         0
embarked    0
dtype: int64

# Encoding as Keras only takes numbers

In [41]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [42]:
X_train_cat_encoded = encoder.fit_transform(
    X_train_cat
)

X_test_cat_encoded = encoder.transform(
    X_test_cat
)

# Scaling

In [43]:
scaler = StandardScaler()

In [44]:
X_train_num_scaled = scaler.fit_transform(
    X_train_num
)

X_test_num_scaled = scaler.transform(
    X_test_num
)

# We reicombine the numeric and cat 

In [50]:
encoded_cols = encoder.get_feature_names_out(categorical_features).tolist()
all_cols = numeric_features + encoded_cols

In [51]:
X_train_processed = pd.DataFrame(np.hstack([
    X_train_num_scaled,
    X_train_cat_encoded]),columns = all_cols
    ,index = X_train.index
)

In [52]:
X_test_processed = pd.DataFrame(np.hstack([
    X_test_num_scaled,
    X_test_cat_encoded
]),columns = all_cols
    ,index = X_test.index)

# However Keras prefer arrays so we will keep the arrys version

In [54]:


X_train_processed_arr = np.hstack([X_train_num_scaled, X_train_cat_encoded])
X_test_processed_arr = np.hstack([X_test_num_scaled, X_test_cat_encoded])

X_train_processed_arr.shape, X_test_processed_arr.shape

((712, 10), (179, 10))

# We start building the ANN

In [55]:
model = keras.Sequential([
    keras.layers.Input(
        shape=(X_train_processed_arr.shape[1],)
        # Keras is indicating how many features a single passenger has
    ),

    # At this point, we define the architecture of the network

    keras.layers.Dense(
        32,  # A hidden layer with 32 neurons
        activation="relu"
    ),

    keras.layers.Dense(
        16,  # A hidden layer with 16 neurons
        activation="relu"  # ReLU allows the network to learn non-linear relationships
    ),

    keras.layers.Dense(
        1,  # One output neuron for the binary classification
        activation="sigmoid"  # Sigmoid converts the output into a probability between 0 and 1
    )
])

# Compile will tell the network how to learn

In [56]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Training

In [57]:
history = model.fit(
    X_train_processed_arr,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32
)

Epoch 1/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6415 - loss: 0.6790 - val_accuracy: 0.7832 - val_loss: 0.6352
Epoch 2/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7627 - loss: 0.6144 - val_accuracy: 0.8182 - val_loss: 0.5907
Epoch 3/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7961 - loss: 0.5656 - val_accuracy: 0.8322 - val_loss: 0.5523
Epoch 4/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8120 - loss: 0.5271 - val_accuracy: 0.8322 - val_loss: 0.5203
Epoch 5/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8014 - loss: 0.4966 - val_accuracy: 0.8042 - val_loss: 0.4934
Epoch 6/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8137 - loss: 0.4708 - val_accuracy: 0.8322 - val_loss: 0.4760
Epoch 7/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8260 - loss: 0.4531 - val_accuracy: 0.8322 - val_loss: 0.4632
Epoch 8/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8243 - loss: 0.4407 - val_accuracy: 0.8322 - val_loss

**epochs**=50

network sees the training dataset up to 50 times

**batch_size**=32

The weights are updated every 32 observations

**validation_split**=0.2

Keras take part of the training for validation


# Model Evaluation

In [58]:
model.evaluate(
    X_test_processed_arr,
    y_test
)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7933 - loss: 0.4466 


[0.4465731382369995, 0.7932960987091064]

**The first number is the binary cross-entropy (loss)**- it tells how far are the probability made by the model from the real values - our case 44% - the lower the better

**Accuracy** - means the model has classifie correctly the 79% of the test passengers

In [59]:
y_pred_probability = model.predict(
    X_test_processed_arr
)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


In [60]:
X_test_processed['probability'] = y_pred_probability
X_test_processed

,age,fare,sibsp,parch,pclass,sex_female,sex_male,embarked_C,embarked_Q,embarked_S,probability
565,-0.426860,-0.159704,1.421753,-0.466183,0.829568,0.0,1.0,0.0,0.0,1.0,0.168363
160,1.109696,-0.327324,-0.465084,0.727782,0.829568,0.0,1.0,0.0,0.0,1.0,0.047446
553,-0.580516,-0.512122,-0.465084,-0.466183,0.829568,0.0,1.0,1.0,0.0,0.0,0.150913
860,0.879212,-0.368795,1.421753,-0.466183,0.829568,0.0,1.0,0.0,0.0,1.0,0.067829
241,-0.081135,-0.339817,0.478335,-0.466183,0.829568,1.0,0.0,0.0,1.0,0.0,0.801761
...,...,...,...,...,...,...,...,...,...,...,...
880,-0.350033,-0.121182,-0.465084,0.727782,-0.370945,1.0,0.0,0.0,0.0,1.0,0.942642
91,-0.734172,-0.499020,-0.465084,-0.466183,0.829568,0.0,1.0,0.0,0.0,1.0,0.123346
883,-0.119549,-0.443929,-0.465084,-0.466183,-0.370945,0.0,1.0,0.0,0.0,1.0,0.097361
473,-0.503688,-0.375388,-0.465084,-0.466183,-0.370945,1.0,0.0,1.0,0.0,0.0,0.969151


# We can Transform the probability in a classs

In [61]:
y_pred = (y_pred_probability >= 0.5).astype(int)
X_test_processed['probability class'] = y_pred
X_test_processed

,age,fare,sibsp,parch,pclass,sex_female,sex_male,embarked_C,embarked_Q,embarked_S,probability,probability class
565,-0.426860,-0.159704,1.421753,-0.466183,0.829568,0.0,1.0,0.0,0.0,1.0,0.168363,0
160,1.109696,-0.327324,-0.465084,0.727782,0.829568,0.0,1.0,0.0,0.0,1.0,0.047446,0
553,-0.580516,-0.512122,-0.465084,-0.466183,0.829568,0.0,1.0,1.0,0.0,0.0,0.150913,0
860,0.879212,-0.368795,1.421753,-0.466183,0.829568,0.0,1.0,0.0,0.0,1.0,0.067829,0
241,-0.081135,-0.339817,0.478335,-0.466183,0.829568,1.0,0.0,0.0,1.0,0.0,0.801761,1
...,...,...,...,...,...,...,...,...,...,...,...,...
880,-0.350033,-0.121182,-0.465084,0.727782,-0.370945,1.0,0.0,0.0,0.0,1.0,0.942642,1
91,-0.734172,-0.499020,-0.465084,-0.466183,0.829568,0.0,1.0,0.0,0.0,1.0,0.123346,0
883,-0.119549,-0.443929,-0.465084,-0.466183,-0.370945,0.0,1.0,0.0,0.0,1.0,0.097361,0
473,-0.503688,-0.375388,-0.465084,-0.466183,-0.370945,1.0,0.0,1.0,0.0,0.0,0.969151,1


# New passenger

In [62]:
new_passenger = pd.DataFrame({
    "age": [30],
    "fare": [50],
    "sibsp": [0],
    "parch": [0],
    "pclass": [2],
    "sex": ["female"],
    "embarked": ["S"]
})

# we need to run all the preprocessing steps

In [63]:
new_num = numeric_imputer.transform(
    new_passenger[numeric_features]
)

new_num_scaled = scaler.transform(new_num)


new_cat = categorical_imputer.transform(
    new_passenger[categorical_features]
)

new_cat_encoded = encoder.transform(new_cat)

new_processed = np.hstack([
    new_num_scaled,
    new_cat_encoded
])

probability = model.predict(
    new_processed
)

probability

C:\Users\matti\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\matti\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step


array([[0.9392417]], dtype=float32)